In [0]:
%pip install transformers datasets yfinance beautifulsoup4 requests pandas matplotlib finbert-embedding

In [0]:
import requests
import time
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

# 1. Configuration
NEWSDATA_KEY = "pub_f0277201240746b5a2d1790087a9efe5" 

# Lokesh's Idea: Map the Ticker to the actual Company Name for the API search
stocks_to_track = [
    {"ticker": "TCS.NS", "query": "Tata Consultancy Services"},
    {"ticker": "RELIANCE.NS", "query": "Reliance Industries"},
    {"ticker": "HDFCBANK.NS", "query": "HDFC Bank"},
    {"ticker": "INFY.NS", "query": "Infosys"},
    {"ticker": "ICICIBANK.NS", "query": "ICICI Bank"},
    {"ticker": "SBIN.NS", "query": "State Bank of India"},
    {"ticker": "BHARTIARTL.NS", "query": "Bharti Airtel"},
    {"ticker": "ITC.NS", "query": "ITC Limited"},
    {"ticker": "LT.NS", "query": "Larsen & Toubro"},
    {"ticker": "BAJFINANCE.NS", "query": "Bajaj Finance"}
]

# 2. Strict Schema for NewsData.io Raw Data
newsdata_schema = StructType([
    StructField("title", StringType(), True),
    StructField("link", StringType(), True),
    StructField("creator", ArrayType(StringType()), True), 
    StructField("description", StringType(), True),
    StructField("pubDate", StringType(), True),
    StructField("source_id", StringType(), True),
    StructField("target_ticker", StringType(), True) 
])

def merge_bronze_news(stocks: list, api_key: str):
    print(f"Starting Bronze API Ingestion for {len(stocks)} stocks...\n")
    
    url = "https://newsdata.io/api/1/news"
    all_articles = []
    
    # 3. Fetch data from API using the mapping
    for stock in stocks:
        ticker_symbol = stock["ticker"]
        company_query = stock["query"]
        
        print(f"Fetching news for: {company_query} ({ticker_symbol})")
        
        params = {
            "apikey": api_key,
            "q": company_query, # <-- Search using the full company name
            "country": "in",
            "category": "business",
            "language": "en"
        }
        
        try:
            response = requests.get(url, params=params)
            response.raise_for_status()
            articles = response.json().get('results', [])
            
            for article in articles:
                # Tag it with the ticker symbol for our database
                article['target_ticker'] = ticker_symbol 
                all_articles.append(article)
                
            time.sleep(1) # Protects against API rate limits
            
        except Exception as e:
            print(f"  -> Error fetching {company_query}: {e}")

    if not all_articles:
        print("No news found for any of the companies in this run.")
        return

    # 4. Create DataFrame
    df_raw = spark.createDataFrame(all_articles, schema=newsdata_schema)
    
    # 5. Map to Bronze Schema (CONTENT COLUMN REMAINS REMOVED)
    df_new_batch = df_raw.select(
        F.col("target_ticker").alias("ticker"),
        F.col("source_id").alias("source"),
        F.concat_ws(", ", F.col("creator")).alias("author"),
        F.col("title").alias("headline"),
        F.col("description"),
        F.col("link").alias("url"),
        F.to_timestamp(F.col("pubDate")).alias("published_at"),
        F.current_timestamp().alias("ingestion_time"),
        F.to_json(F.struct("*")).alias("raw_json")
    ).dropDuplicates(["url"]) 

    # 6. Merge into Delta Table to prevent duplicates
    table_name = "finance.analysis.bronze_news"
    
    if spark.catalog.tableExists(table_name):
        delta_table = DeltaTable.forName(spark, table_name)
        print("\nChecking against existing database to prevent duplicates...")
        
        delta_table.alias("target").merge(
            df_new_batch.alias("source"),
            "target.url = source.url"
        ).whenNotMatchedInsertAll().execute() 
        
        print(f"✅ Merge complete! Only BRAND NEW articles were added to {table_name}.")
        
    else:
        print(f"\nTable {table_name} not found. Creating it now...")
        df_new_batch.write.format("delta").saveAsTable(table_name)
        print(f"✅ Table created and initial batch saved.")

# --- EXECUTE ---
merge_bronze_news(stocks_to_track, NEWSDATA_KEY)

In [0]:
import pandas as pd
from transformers import pipeline
from pyspark.sql import functions as F

def simple_incremental_silver_layer():
    print("1. Reading from Bronze Layer...")
    bronze_table = "finance.analysis.bronze_news"
    silver_table = "finance.analysis.silver_news"
    
    # Read Bronze
    df_bronze = spark.read.table(bronze_table).filter(F.col("headline").isNotNull())
    
    # Find new rows only
    if spark.catalog.tableExists(silver_table):
        df_silver_existing = spark.read.table(silver_table)
        df_new = df_bronze.join(df_silver_existing, on=["ticker", "headline"], how="left_anti")
    else:
        df_new = df_bronze

    new_count = df_new.count()
    if new_count == 0:
        print("No new articles to process. Silver Layer is up to date!")
        return

    print(f"2. Pulling {new_count} rows to the main node...")
    # THE BIG CHANGE: Convert to Pandas to bypass PySpark worker limitations
    pdf_new = df_new.toPandas()

    print("3. Loading FinBERT locally...")
    # Standard Python HuggingFace pipeline (No hacks needed!)
    sentiment_pipeline = pipeline("sentiment-analysis", model="ProsusAI/finbert")
    
    print("4. Scoring headlines (Optimized Batch Mode)...")
    # Clean all text into a list
    text_list = [str(text)[:512] if text else "neutral" for text in pdf_new['headline'].tolist()]
    
    # Feed the entire list to FinBERT at once for maximum speed
    batch_results = sentiment_pipeline(text_list)
    
    labels = []
    scores = []
    
    for output in batch_results:
        label = output['label'].lower()
        if label == 'positive': scores.append(1.0)
        elif label == 'negative': scores.append(-1.0)
        else: scores.append(0.0)
        labels.append(label)

    # Add the new columns back to our Pandas DataFrame
    pdf_new['sentiment_label'] = labels
    pdf_new['sentiment_score'] = scores
    
    print("5. Saving back to Delta Table...")
    # Convert back to a Spark DataFrame
    df_scored = spark.createDataFrame(pdf_new)
    
    # Clean up columns to match your exact schema
    df_silver_final = df_scored.select(
        F.col("ticker"),
        F.col("headline"),
        F.col("source"),
        F.col("published_at").alias("publish_time"), 
        F.col("sentiment_label"),
        F.col("sentiment_score"),
        F.current_timestamp().alias("processed_time")
    )
    
    if spark.catalog.tableExists(silver_table):
        df_silver_final.write.format("delta").mode("append").saveAsTable(silver_table)
    else:
        df_silver_final.write.format("delta").mode("overwrite").saveAsTable(silver_table)

    print(f"✅ Success! {new_count} articles scored and saved.")

# --- EXECUTE ---
simple_incremental_silver_layer()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def build_normalized_gold_layer():
    print("1. Reading and Normalizing Silver Tickers...")
    silver_table = "finance.analysis.silver_news"
    gold_table = "finance.analysis.gold_sentiment_report"
    
    df_silver = spark.read.table(silver_table)
    
    # --- THE NORMALIZATION FIX ---
    # This splits 'TCS.NS' at the '.' and takes the first part 'TCS'
    # It also forces everything to UPPERCASE just in case.
    df_clean = df_silver.withColumn(
        "normalized_ticker", 
        F.upper(F.split(F.col("ticker"), "\.").getItem(0))
    )

    print("2. Aggregating Merged Tickers...")
    # We now group by the 'normalized_ticker' instead of the raw 'ticker'
    df_metrics = df_clean.groupBy("normalized_ticker").agg(
        F.round(F.avg("sentiment_score"), 3).alias("sentiment_score"),
        F.count("headline").alias("article_volume"),
        F.max("publish_time").alias("last_updated")
    ).withColumnRenamed("normalized_ticker", "ticker")
    
    # Add the Bullish/Bearish Verdict
    df_verdict = df_metrics.withColumn(
    "market_verdict",
    F.when(F.col("sentiment_score") >= 0.15, "Bullish")
     .when(F.col("sentiment_score") <= -0.10, "Bearish") # This will catch your -0.143
     .otherwise("Neutral")
    )
    
    print("3. Extracting Top Drivers for Merged Data...")
    window_pos = Window.partitionBy("normalized_ticker").orderBy(F.desc("sentiment_score"))
    df_top_pos = df_clean.filter(F.col("sentiment_score") > 0) \
        .withColumn("rank", F.row_number().over(window_pos)) \
        .filter(F.col("rank") == 1) \
        .select(F.col("normalized_ticker").alias("ticker"), F.col("headline").alias("top_positive_driver"))
        
    window_neg = Window.partitionBy("normalized_ticker").orderBy(F.asc("sentiment_score"))
    df_top_neg = df_clean.filter(F.col("sentiment_score") < 0) \
        .withColumn("rank", F.row_number().over(window_neg)) \
        .filter(F.col("rank") == 1) \
        .select(F.col("normalized_ticker").alias("ticker"), F.col("headline").alias("top_negative_driver"))
        
    print("4. Final Assembly...")
    df_gold_final = df_verdict \
        .join(df_top_pos, on="ticker", how="left") \
        .join(df_top_neg, on="ticker", how="left") \
        .orderBy(F.desc("sentiment_score")) \
        .fillna("No specific catalysts found.")
    
    print(f"5. Saving to {gold_table}...")
    df_gold_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(gold_table)
    
    print("✅ Normalization Complete! Merged '.NS' and raw tickers successfully.\n")
    df_gold_final.show(truncate=False)

# --- EXECUTE ---
build_normalized_gold_layer()

In [0]:
%sql
DROP TABLE IF EXISTS finance.analysis.gold_sentiment_report;